# Autonomous Farming: Corn Row Alignment Scoring
We built a machine learning system that looks at top-down photos of a corn field, finds every individual corn plant, measures *where* each plant is and *which way it's leaning (growing)*, and then scores how well each plant is aligned with the row it was planted in. The output is two numbers per plant (a positional error in pixels and an angular error in degrees) rolled up into a 0–100 alignment score, with per-row averages.

## The five deliverables, mapped to sections of this notebook
| # | Deliverable | Where in notebook |
|---|-------------|-------------------|
| 1 | Detect individual corn plants in an image | §4 Training, §5 Evaluation |
| 2 | Estimate position and orientation of each plant | §6 Inference |
| 3 | Define the optimal growth line per row | §8 Optimal line |
| 4 | Compute the deviation from that line | §9 Scoring |
| 5 | Produce row-level and plant-level alignment scores | §9 Scoring, §11 Batch export |

## How it works, at a system level
1. **Camera → pixels.** A top-down image of the field.
2. **Detection model → list of plants.** A neural network (YOLO11-pose) looks at the image and returns, for each plant it finds, a bounding box plus three landmark points: the stem center, the tip of the left leaf, and the tip of the right leaf.
3. **Post-processing → measurements.** We take those three points and compute (a) the plant's position = stem location, and (b) the plant's orientation = direction of the line through the two leaf tips.
4. **Row fitting → reference lines.** We group plants by row and either use a known reference line (if we know exactly where the seeds were planted) or we fit a best-fit line through the detected stems.
5. **Scoring → numbers.** For each plant: how far is it from its row's reference line? How rotated is its leaf axis from the expected orientation? We combine these into a single 0–100 score.

---
# 1. Setup

Before doing anything interesting, we need to:

1. Confirm the GPU is working. Training on a CPU would take days.
2. Set the project folder so all outputs land in one place.
3. Import the libraries we'll use throughout.

## Before running this notebook, install these packages (once, from your terminal):

```
pip install ultralytics scikit-learn matplotlib opencv-python
```

If you see a CUDA / GPU-related error later, you may also need the GPU build of PyTorch:

```
pip uninstall torch torchvision -y
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch  # PyTorch — the deep learning framework YOLO is built on

# torch.cuda.is_available() returns True if an NVIDIA GPU + compatible drivers are installed.
# The `assert` line crashes loudly if this returns False, so you find out NOW, not 3 hours into training.
assert torch.cuda.is_available(), "No GPU detected. Check CUDA install / PyTorch version."

# Print GPU model name (useful to confirm you're using the right one if you have multiple).
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
import os
from pathlib import Path
PROJECT_ROOT = Path(r"C:\Users\atpou\projects\T1_Corn_Alignment")

# Create the folder if it doesn't already exist. exist_ok=True means "don't error if it's already there".
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Change the working directory so relative paths in later cells resolve here.
os.chdir(PROJECT_ROOT)

print(f"Working from: {PROJECT_ROOT}")

In [ ]:
import json          # read/write JSON files
import shutil        # copy / delete directories
import random        # Python's built-in random number generator
import yaml          # read/write YAML files (used for YOLO's config format)
from collections import defaultdict  # a dict that auto-creates missing keys as empty lists — handy for grouping
import numpy as np              # arrays and linear algebra — the workhorse of anything numerical in Python
import cv2                      # OpenCV — image loading, drawing, classical computer vision
import matplotlib.pyplot as plt # plotting — how we visualize images inline
from sklearn.cluster import DBSCAN  # clustering algorithm we'll use for row assignment

SEED = 42
random.seed(SEED)         # seed Python's random
np.random.seed(SEED)      # seed NumPy's random
torch.manual_seed(SEED)   # seed PyTorch's random (for weight init, augmentations, etc.)

# Constants describing our annotation scheme: 3 keypoints per plant, in a fixed order.
NUM_KEYPOINTS = 3
KP_STEM  = 0   # index 0 in the keypoint list = stem center
KP_LEFT  = 1   # index 1 = leaf-tip-left
KP_RIGHT = 2   # index 2 = leaf-tip-right

---
# 2. Dataset preparation
The dataset consists of a folder of corn photos, each paired with a text file that has one line per plant: where the bounding box is, and where the three keypoints are. Both images and labels get split into `train/` and `valid/`

## Annotation format
The dataset is exported from Roboflow in "YOLOv8 Pose" format. Each label file looks like:

```
0  0.437 0.389 0.300 0.183  0.479 0.383 2   0.558 0.303 2   0.303 0.316 2
|  |____bbox (x,y,w,h)____| |_kp0 stem_|   |_kp1 left_|   |_kp2 right|
|                                          visibility flag (2 = visible, 0 = not annotated)
class ID (0 = corn)
```

All coordinates are **normalized** to [0, 1], making the labels resolution-independent.

## Expected folder structure

Unzip the Roboflow export into `PROJECT_ROOT/raw_dataset/` so it looks like this:

```
raw_dataset/
|-- train/images/*.jpg   <- training images
|-- train/labels/*.txt   <- one label file per image, same base name
|-- valid/images/*.jpg   <- validation images (model never trains on these)
|-- valid/labels/*.txt
|-- data.yaml            <- we ignore this and generate our own in section 3
```

Now, before proceeding, we need to:
1. Point to the raw dataset and count what we have
2. Canonicalize keypoint order
3. Eyeball a few samples (sanity check)

In [ ]:
# Where the raw Roboflow export lives
RAW_DATASET = PROJECT_ROOT / 'raw_dataset'
assert RAW_DATASET.exists(), f"Unzip your Roboflow export into {RAW_DATASET}"

# Walk the expected subfolders and print counts
for split in ['train', 'valid', 'test']:   # 'test' is optional — many exports skip it
    img_dir = RAW_DATASET / split / 'images'
    lbl_dir = RAW_DATASET / split / 'labels'
    if img_dir.exists():
        # .glob('*') returns every file; len(list(...)) counts them
        n_img = len(list(img_dir.glob('*')))
        n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        print(f"{split:6s}  images={n_img:4d}  labels={n_lbl:4d}")
        # If images and labels don't match, something is wrong with the export
        if n_img != n_lbl:
            print(f"  WARNING: image count and label count don't match for {split}!")

In [ ]:
def canonicalize_label_file(src_path: Path, dst_path: Path):
    """
    Read one YOLO pose label file and rewrite it with:
      - Keypoint 0 = the point closest to the bbox center (the stem)
      - Keypoint 1 = the remaining point with smaller x (left)
      - Keypoint 2 = the remaining point with larger x  (right)
    """
    out_lines = []  # we'll build the new file line by line

    with open(src_path) as f:  # open the source label file for reading
        for line in f:
            # Each line has: class cx cy w h kp0x kp0y kp0v kp1x kp1y kp1v kp2x kp2y kp2v
            parts = line.strip().split()

            # Sanity check: do we have enough numbers? (1 class + 4 bbox + 3*3 keypoints = 14)
            if len(parts) < 5 + NUM_KEYPOINTS * 3:
                out_lines.append(line.rstrip())  # malformed line — keep as-is, don't crash
                continue

            cls = parts[0]  # class ID as string ('0' for corn)
            cx, cy, w, h = map(float, parts[1:5])  # bounding box center (cx,cy) and size (w,h)

            # Parse the three keypoints into (x, y, visibility) tuples
            kps = []
            for i in range(NUM_KEYPOINTS):
                base = 5 + i * 3    # where this keypoint's 3 numbers start in the line
                x = float(parts[base])
                y = float(parts[base + 1])
                v = float(parts[base + 2])
                kps.append((x, y, v))

            # Only consider keypoints that are actually annotated (visibility > 0).
            # Sometimes a leaf is hidden and the annotator marks v=0 to skip it.
            visible = [(i, k) for i, k in enumerate(kps) if k[2] > 0]
            if len(visible) == 0:
                out_lines.append(line.rstrip())  # nothing annotated — leave it
                continue

            # --- Step 1: find the stem = whichever keypoint is closest to the bbox center ---
            def dist_to_centroid(k):
                # squared distance (no need for sqrt since we only compare distances)
                return (k[0] - cx) ** 2 + (k[1] - cy) ** 2

            # min() with a key function picks the item with the smallest value of that function
            stem_idx, stem_kp = min(visible, key=lambda iv: dist_to_centroid(iv[1]))

            # The other two keypoints are the tips
            tip_items = [(i, k) for i, k in enumerate(kps) if i != stem_idx]

            # --- Step 2: sort tips by x-coordinate so left is always index 1 ---
            def tip_sort_key(ik):
                _, k = ik
                # If a tip isn't annotated, push it to the end with a large sentinel value
                if k[2] == 0:
                    return float('inf')
                return k[0] - stem_kp[0]  # x relative to the stem

            tip_items.sort(key=tip_sort_key)
            left_kp  = tip_items[0][1]  # smaller x → left tip
            # Handle the edge case where only one tip was annotated
            right_kp = tip_items[1][1] if len(tip_items) > 1 else (0.0, 0.0, 0.0)

            # --- Reassemble in canonical order: [stem, left, right] ---
            new_kps = [stem_kp, left_kp, right_kp]
            kp_str = ' '.join(f"{x:.6f} {y:.6f} {int(v)}" for (x, y, v) in new_kps)
            out_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} {kp_str}")

    # Write the cleaned file to its destination
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with open(dst_path, 'w') as f:
        f.write('\n'.join(out_lines) + '\n')


# We write the cleaned dataset into a separate folder so the raw export is untouched.
# This means you can re-run this cell safely without losing the original labels.
DATASET_ROOT = PROJECT_ROOT / 'dataset'

# If the cleaned dataset already exists from a previous run, delete it first.
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)

# Walk each split (train/valid/test) and process every label file
for split in ['train', 'valid', 'test']:
    src_img = RAW_DATASET / split / 'images'
    src_lbl = RAW_DATASET / split / 'labels'
    if not src_img.exists():  # skip splits that don't exist (e.g. if no test set)
        continue

    dst_img = DATASET_ROOT / split / 'images'
    dst_lbl = DATASET_ROOT / split / 'labels'
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    # Copy images (not symlinks, because Windows symlinks require admin permissions)
    for img in src_img.iterdir():
        tgt = dst_img / img.name
        if not tgt.exists():
            shutil.copy2(img, tgt)  # copy2 preserves metadata like modification time

    # Process and rewrite every label file
    for lbl in src_lbl.glob('*.txt'):
        canonicalize_label_file(lbl, dst_lbl / lbl.name)

    print(f"{split:6s}  done")

In [ ]:
def draw_annotation(img, label_path):
    """
    Takes an image and its label file, and returns a copy of the image
    with the bounding box and keypoints drawn on top.
    Used for visual debugging only.
    """
    H, W = img.shape[:2]  # image height, width in pixels
    out = img.copy()      # don't modify the original

    # BGR colors (OpenCV convention) for each keypoint
    COLORS = {
        KP_STEM:  (255, 0, 0),    # red
        KP_LEFT:  (0, 200, 255),  # cyan-ish
        KP_RIGHT: (255, 200, 0),  # orange-ish
    }
    NAMES = {KP_STEM: 'stem', KP_LEFT: 'L', KP_RIGHT: 'R'}

    if not label_path.exists():
        return out  # no labels — just return the raw image

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5 + NUM_KEYPOINTS * 3:
                continue

            # Convert normalized bbox [0,1] back to absolute pixels for drawing
            cx, cy, w, h = map(float, parts[1:5])
            x1 = int((cx - w/2) * W)
            y1 = int((cy - h/2) * H)
            x2 = int((cx + w/2) * W)
            y2 = int((cy + h/2) * H)
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 255, 0), 2)  # green bbox

            # Draw each keypoint
            for i in range(NUM_KEYPOINTS):
                base = 5 + i * 3
                kx = float(parts[base]) * W      # denormalize x
                ky = float(parts[base + 1]) * H  # denormalize y
                v = float(parts[base + 2])       # visibility flag
                if v > 0:  # only draw if actually annotated
                    cv2.circle(out, (int(kx), int(ky)), 5, COLORS[i], -1)
                    cv2.putText(out, NAMES[i], (int(kx) + 6, int(ky) - 6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS[i], 1)
    return out


# Pick 6 random training images to spot-check
train_imgs = sorted((DATASET_ROOT / 'train' / 'images').iterdir())
sample = random.sample(train_imgs, min(6, len(train_imgs)))

# Set up a 2x3 grid of subplots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.ravel(), sample):
    # OpenCV loads images as BGR; matplotlib expects RGB — convert
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    lbl = DATASET_ROOT / 'train' / 'labels' / (img_path.stem + '.txt')
    ax.imshow(draw_annotation(img, lbl))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.suptitle("Sanity check: bbox + canonicalized keypoints", fontsize=13)
plt.tight_layout()
plt.show()

---
# 3. YOLO Configuration File

## What is `data.yaml`?

YOLO reads training configuration from a small text file in YAML format. We tell it three critical things:

| Setting | What it means |
|---------|---------------|
| `path`, `train`, `val` | Where the images are |
| `kpt_shape: [3, 3]` | Each plant has 3 keypoints, each with 3 values (x, y, visibility) |
| `flip_idx: [0, 2, 1]` | When training flips an image horizontally, remap keypoints: stem stays (0), left becomes right (1→2) and right becomes left (2→1) |
| `names: ['corn']` | We only have one class — "corn" |

## Why `flip_idx` matters

Horizontal flipping is a free way to double the effective size of your dataset — a mirrored corn plant is still a valid corn plant. But if you flip the image without also swapping the L/R keypoint labels, the model sees what's visually a "left leaf" labeled as "right" — broken supervision. `flip_idx` tells YOLO to swap them automatically.

In [ ]:
# Compose the config dictionary in Python first, then dump it to YAML
data_yaml_path = DATASET_ROOT / 'data.yaml'

data_cfg = {
    # Absolute path to the dataset root
    'path': str(DATASET_ROOT),
    # Relative paths to train and val image folders (YOLO finds the matching labels automatically)
    'train': 'train/images',
    'val':   'valid/images',
    # Optional test split — include only if it exists
    'test':  'test/images' if (DATASET_ROOT / 'test' / 'images').exists() else None,

    # Keypoint shape: [num_keypoints, num_values_per_keypoint]
    # 3 keypoints, each with (x, y, visibility) = 3 values
    'kpt_shape': [NUM_KEYPOINTS, 3],

    # When horizontal flip augmentation triggers, this list says:
    # "keypoint index 0 becomes index 0 (stem — unchanged)"
    # "keypoint index 1 becomes index 2 (left tip → right tip)"
    # "keypoint index 2 becomes index 1 (right tip → left tip)"
    'flip_idx':  [KP_STEM, KP_RIGHT, KP_LEFT],

    # Just one class — no weeds, no other crops (for this project)
    'names': ['corn'],
}

# Remove any entries where the value is None (dict comprehension — filters {k:v for ...})
data_cfg = {k: v for k, v in data_cfg.items() if v is not None}

# Write the YAML file
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

# Print so you can see what's in it
print(open(data_yaml_path).read())

---
# 4. YOLO11-pose model Training

## What we're actually training

YOLO is a family of object detection neural networks. YOLO**-pose** variants extend this to also predict keypoints within each detected object — exactly what we need.

We start from a pre-trained model (weights downloaded automatically — trained by Ultralytics on the COCO dataset of 100K+ human pose images) and **fine-tune** it on our 500 corn images. This is transfer learning: the model already knows how to "look" at images and find objects; we're teaching it to specialize in finding corn.

## Why train two sizes

| Model | Params | Speed | Accuracy |
|-------|--------|-------|----------|
| YOLO11n-pose | 3M | Fast | Good |
| YOLO11s-pose | 10M | Slower | Better |

Nano is fast enough for real-time use on embedded hardware. Small is more accurate but ~3× larger. Training both lets us pick the right trade-off once we see how well each performs.

## Key training hyperparameters

| Parameter | Meaning |
|-----------|---------|
| `epochs=200` | Look at the whole dataset 200 times |
| `imgsz=640` | Resize every image to 640×640 before training |
| `batch=16` | Process 16 images at a time (lower if you run out of GPU memory) |
| `patience=40` | Stop early if validation score hasn't improved in 40 epochs (saves time) |
| `pose=30.0` | Weight the keypoint-accuracy loss heavily (default is 12) — keypoints matter more than bbox for us |

## 4.1 Weather-robust augmentation

### What is augmentation?

During training, YOLO randomly modifies each image before feeding it to the model: rotate it, change the colors, add random noise. This teaches the model to be invariant to these variations — it learns what "corn" looks like regardless of lighting, angle, etc.

### Why we use aggressive settings

The project brief calls out rain and sunshine as deployment concerns. We can't collect data in every weather condition, but we can simulate them via augmentation. Strong brightness and HSV jitter simulate sunlight direction and intensity. Random erasing simulates leaves partially occluding each other, rain streaks, or dirt on the camera.

In [ ]:
# Augmentation parameters passed to YOLO's trainer.
# Values are stronger than defaults because our dataset is small and we want robustness.
AUG = dict(
    # Color jitter — simulates different sun angles / times of day
    hsv_h=0.02,        # hue shift (±2%) — small, otherwise plants stop looking green
    hsv_s=0.70,        # saturation (±70%) — strong, simulates overcast vs. direct sun
    hsv_v=0.50,        # brightness (±50%) — strong, simulates shadow vs. bright sun

    # Geometric jitter — simulates camera not being perfectly level/centered
    degrees=15,        # rotation up to ±15°
    translate=0.10,    # translation up to ±10% of image size
    scale=0.40,        # scale up to ±40% — accounts for varying camera height
    shear=3,           # shear up to ±3° — tiny warping
    perspective=0.0005,  # perspective distortion — very small

    # Flipping
    fliplr=0.5,        # 50% chance of horizontal flip (uses flip_idx to swap L/R keypoints)
    flipud=0.0,        # never vertical flip — top-down corn has a consistent orientation

    # Advanced augmentation
    mosaic=1.0,        # always combine 4 training images into one — excellent for small datasets
    mixup=0.10,        # 10% chance to blend two images — regularization
    erasing=0.30,      # 30% chance to erase a random rectangle — simulates occlusion
)
# Dictionary variable that gets unpacked into model.train(...) in the next cells.

## 4.2 Train YOLO11n-pose

This is the fast model (expect 1 hour of training on a decent GPU). Watch the output, you'll see training loss decreasing each epoch, and validation metrics (mAP50, pose mAP50) increasing. If training loss decreases but validation stays flat, the model is memorizing instead of learning - but the augmentation and early stopping should prevent that.

In [ ]:
from ultralytics import YOLO  # YOLO is the main class we use for training and inference

# Load the nano pose model with pre-trained weights.
# If 'yolo11n-pose.pt' isn't already downloaded, Ultralytics fetches it the first time.
model_n = YOLO('yolo11n-pose.pt')

# Kick off training.
# **AUG unpacks our augmentation dict into keyword arguments.
results_n = model_n.train(
    data=str(data_yaml_path),   # path to data.yaml we wrote in section 3
    epochs=200,                 # max training passes through the dataset
    imgsz=640,                  # input resolution
    batch=16,                   # mini-batch size (reduce to 8 or 4 if you see CUDA OOM errors)
    patience=40,                # early-stop after 40 epochs of no improvement
    device=0,                   # GPU index (0 = first GPU)
    project=str(PROJECT_ROOT / 'runs'),  # where to save logs/weights
    name='yolo11n_pose_corn',   # subfolder name for this run
    exist_ok=True,              # overwrite if the run folder already exists
    pose=30.0,                  # keypoint regression loss weight (default 12)
    kobj=2.0,                   # keypoint objectness loss weight
    **AUG,                      # unpack augmentation settings
)
# When this finishes, you'll have weights at: runs/yolo11n_pose_corn/weights/best.pt

## 4.3 Train YOLO11s-pose

Same as above but with a larger model. It is around 2-3× slower to train. Skip this cell if you're short on time - nano is usually good enough.

In [ ]:
# Same approach, bigger model
model_s = YOLO('yolo11s-pose.pt')
results_s = model_s.train(
    data=str(data_yaml_path),
    epochs=200,
    imgsz=640,
    batch=16,                # if you hit out-of-memory errors, try batch=8
    patience=40,
    device=0,
    project=str(PROJECT_ROOT / 'runs'),
    name='yolo11s_pose_corn',
    exist_ok=True,
    pose=30.0,
    kobj=2.0,
    **AUG,
)

---
# 5. Trained Model Evaluation
We look at two kinds of metrics:

## Detection-quality metrics (mAP)

**mAP = mean Average Precision**, it asks: "across all IoU thresholds (how much bbox overlap counts as a correct detection), how many plants did the model find correctly and at what confidence?"

- **mAP50** — lenient, counts a detection as correct if IoU ≥ 50%. Typical "works well" threshold: > 0.85.
- **mAP50-95** — strict, averages over IoU thresholds from 50% to 95%. Typical "works well": > 0.60.

Both exist for boxes and for keypoints. Keypoint mAP is more relevant for us since the downstream alignment math lives on the keypoints.

We will use validation mAP for YOLO to track the best model seen so far and save it as best.pt. That's what will evaluate further with Pixel error.

## Per-keypoint pixel error (direct, interpretable)

For each validation image, we run the model, match each prediction to its ground-truth by bounding-box overlap, and compute the pixel distance between predicted and actual keypoint locations. Averaging over all matched plants gives us interpretable numbers.

### What to look for

- **Stem error** should be the smallest — it's near the plant center, visually unambiguous.
- **Leaf tip errors** tend to be larger — leaves are thin, and exact "tip" can be ambiguous.
- **p95 (95th percentile)** tells you worst-case behavior. If median is 3 pixels but p95 is 40 pixels, you have rare catastrophic failures you should inspect.

As a rule of thumb: stem pixel error should be less than `POS_TOLERANCE_PX / 3` (we set `POS_TOLERANCE_PX = 30` in section 9, so aim for stem error < 10 pixels) to keep scoring noise small compared to real planting noise.

In [ ]:
RUNS_DIR = PROJECT_ROOT / 'runs'
BEST_N = RUNS_DIR / 'yolo11n_pose_corn' / 'weights' / 'best.pt'
BEST_S = RUNS_DIR / 'yolo11s_pose_corn' / 'weights' / 'best.pt'

# Prefer the `s` model if we trained it; otherwise fall back to `n`.
BEST = BEST_S if BEST_S.exists() else BEST_N
print(f"Evaluating: {BEST}")

# Reload the best checkpoint (we could also keep using `model_n`/`model_s` directly;
# this makes the cell self-contained so you can re-run evaluation without re-training).
model = YOLO(str(BEST))

# model.val() runs one pass over the validation set and returns all the standard metrics.
# conf=0.001 means accept all predictions (we compute precision/recall across the whole confidence range).
# iou=0.6 is the threshold for non-max suppression (merging overlapping detections).
metrics = model.val(data=str(data_yaml_path), imgsz=640, conf=0.001, iou=0.6)

print(f"\nBox mAP50:      {metrics.box.map50:.3f}  (higher is better, >0.85 is good)")
print(f"Box mAP50-95:   {metrics.box.map:.3f}    (higher is better, >0.60 is good)")
print(f"Pose mAP50:     {metrics.pose.map50:.3f} (keypoint localization quality)")
print(f"Pose mAP50-95:  {metrics.pose.map:.3f}")

In [ ]:
def iou_xyxy(a, b):
    """
    Compute Intersection-over-Union between two axis-aligned bounding boxes.
    Each box is [x1, y1, x2, y2] (top-left and bottom-right corners).
    Returns a value in [0, 1]. Higher = more overlap = more likely the same object.
    """
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    # Intersection box
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1); ih = max(0, iy2 - iy1)
    inter = iw * ih

    # Union = sum of areas - intersection
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0


# Accumulate per-keypoint pixel errors across the whole validation set
val_imgs = sorted((DATASET_ROOT / 'valid' / 'images').iterdir())
errors = {KP_STEM: [], KP_LEFT: [], KP_RIGHT: []}

for img_path in val_imgs:
    img = cv2.imread(str(img_path))
    H, W = img.shape[:2]

    # Load ground truth for this image
    lbl_path = DATASET_ROOT / 'valid' / 'labels' / (img_path.stem + '.txt')
    if not lbl_path.exists():
        continue

    # Parse GT boxes and keypoints (denormalize to pixels)
    gts = []
    with open(lbl_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5 + NUM_KEYPOINTS * 3:
                continue
            cx, cy, w, h = map(float, p[1:5])
            box = [(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H]
            kps = []
            for i in range(NUM_KEYPOINTS):
                base = 5 + i * 3
                kps.append((float(p[base]) * W, float(p[base + 1]) * H, float(p[base + 2])))
            gts.append((box, kps))

    # Run the model
    preds = model.predict(str(img_path), conf=0.25, iou=0.5, verbose=False)[0]
    if preds.boxes is None or len(preds.boxes) == 0:
        continue  # no detections — all GTs are "missed"; skip for error calc

    pred_boxes = preds.boxes.xyxy.cpu().numpy()     # (N, 4) array of boxes in pixels
    pred_kps   = preds.keypoints.xy.cpu().numpy()   # (N, 3, 2) array of keypoints in pixels

    # For each GT plant, match to the predicted plant with highest IoU
    for gt_box, gt_kps in gts:
        ious = [iou_xyxy(gt_box, pb) for pb in pred_boxes]
        if not ious or max(ious) < 0.3:
            continue  # no prediction overlapped this GT enough — treat as missed
        best = int(np.argmax(ious))  # index of the best-matching prediction

        # For each keypoint, accumulate the pixel distance between GT and prediction
        for i in range(NUM_KEYPOINTS):
            gx, gy, gv = gt_kps[i]
            if gv == 0:
                continue  # GT didn't annotate this keypoint — skip
            px, py = pred_kps[best, i]
            errors[i].append(np.hypot(px - gx, py - gy))  # Euclidean distance

# Print summary
print(f"{'Keypoint':<14}{'mean px':>10}{'median px':>12}{'p95 px':>10}{'n':>8}")
for i, name in zip([KP_STEM, KP_LEFT, KP_RIGHT], ['stem', 'leaf-left', 'leaf-right']):
    e = np.array(errors[i])
    if len(e):
        print(f"{name:<14}{e.mean():>10.2f}{np.median(e):>12.2f}"
              f"{np.percentile(e, 95):>10.2f}{len(e):>8}")

In [ ]:
# --Sanity Check--
# Look at 4 random validation images with the model's predictions drawn on.

sample_val = random.sample(val_imgs, min(4, len(val_imgs)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, img_path in zip(axes.ravel(), sample_val):
    # model.predict returns a Results object; [0] = results for image 0 (we pass one image)
    result = model.predict(str(img_path), conf=0.25, verbose=False)[0]
    # .plot() returns a numpy array with boxes + keypoints already drawn
    annotated = result.plot(kpt_radius=6, kpt_line=True)
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')
plt.suptitle("Model predictions on validation images", fontsize=13)
plt.tight_layout()
plt.show()

---
# 6. Inference Wrapper

The raw output of `model.predict()` is a complex object with tensors, confidence scores, etc. For the downstream code (row assignment, scoring), we want a simple list of dictionaries - one per plant - with named fields.

This cell defines a single function `detect_plants(image)` that returns a list like:

```python
[
    {
        'bbox':           [x1, y1, x2, y2],       # in pixels
        'confidence':     0.94,                    # how sure the model is (0 to 1)
        'stem':           (450.2, 312.1),          # (x, y) or None if not confident
        'left_tip':       (420.0, 295.3),
        'right_tip':      (485.7, 301.8),
        'leaf_axis_deg':  12.5,                    # angle of the left-right axis
    },
    ...
]
```

This is the clean interface every later cell uses.

In [ ]:
# Confidence threshold for keypoints: below this, we treat the keypoint as "missing"
# rather than trusting a low-confidence guess. Tune based on section 5.2 results.
KP_CONF_THRESHOLD = 0.3


def axis_angle_deg(p_left, p_right):
    """
    Compute the angle of the axis through the two leaf tips.

    Returns angle in degrees in the range (-90, 90]:
      0 deg   = horizontal (tips are left-right)
      90 deg  = vertical   (one tip above the other)
      -45 deg = diagonal down-right

    Why (-90, 90] instead of (-180, 180]? Because an axis (a line) has no direction.
    An axis at +170 deg and an axis at -10 deg describe the same line.
    We normalize so downstream code doesn't have to worry about equivalent angles.
    """
    if p_left is None or p_right is None:
        return None  # can't compute an axis if either tip is missing

    dx = p_right[0] - p_left[0]
    dy = p_right[1] - p_left[1]
    # arctan2 gives signed angle in (-180, 180]; np.degrees converts radians to degrees
    angle = np.degrees(np.arctan2(dy, dx))

    # Wrap into (-90, 90] — flip by 180 deg if outside that range (same line, opposite direction)
    while angle >   90: angle -= 180
    while angle <= -90: angle += 180
    return angle


def detect_plants(image_or_path, conf=0.25, iou=0.5):
    """
    Run the trained model on one image and return a list of plant dictionaries.

    Parameters
    ----------
    image_or_path : str or np.ndarray
        Path to an image file, or an image array already loaded with cv2.
    conf : float
        Minimum confidence to keep a detection (0 to 1). 0.25 = keep everything that's
        25%+ likely to be a plant. Lower catches more, but with more false positives.
    iou : float
        IoU threshold for non-max suppression (merging overlapping detections).
    """
    # Run inference; [0] because predict returns a list (one per input image)
    r = model.predict(image_or_path, conf=conf, iou=iou, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []  # no plants detected

    # Pull tensors off the GPU and convert to numpy arrays for convenience
    boxes = r.boxes.xyxy.cpu().numpy()   # (N, 4) — each row is [x1, y1, x2, y2]
    confs = r.boxes.conf.cpu().numpy()   # (N,)   — confidence per detection
    kp_xy = r.keypoints.xy.cpu().numpy()  # (N, 3, 2) — (x, y) for each of 3 keypoints
    # Per-keypoint confidence isn't always exposed; fall back to 1.0 if unavailable
    try:
        kp_conf = r.keypoints.conf.cpu().numpy()  # (N, 3)
    except Exception:
        kp_conf = np.ones((len(boxes), NUM_KEYPOINTS))

    # Build our simple list of dictionaries
    plants = []
    for i in range(len(boxes)):

        def kp_or_none(idx):
            """Return keypoint (x,y) tuple or None if confidence is too low."""
            if kp_conf[i, idx] < KP_CONF_THRESHOLD:
                return None
            return (float(kp_xy[i, idx, 0]), float(kp_xy[i, idx, 1]))

        stem  = kp_or_none(KP_STEM)
        left  = kp_or_none(KP_LEFT)
        right = kp_or_none(KP_RIGHT)

        plants.append({
            'bbox':          boxes[i].tolist(),
            'confidence':    float(confs[i]),
            'stem':          stem,
            'left_tip':      left,
            'right_tip':     right,
            'leaf_axis_deg': axis_angle_deg(left, right),
        })
    return plants


# --- Quick demo ---
demo_path = random.choice(val_imgs)
plants = detect_plants(str(demo_path))
print(f"Detected {len(plants)} plants in {demo_path.name}")
# Print the first 3 plants to confirm the structure
for p in plants[:3]:
    print(f"  conf={p['confidence']:.2f}  stem={p['stem']}  axis={p['leaf_axis_deg']}")

---
# 7. Per-row Seeding Line Builder - for live camera input

## Deployment model

This notebook is the prototype for the deployed system: a downward-facing camera (mounted on a tractor, or held by an operator) moves along the seeding line, capturing video frames in real time. The pipeline produces per-plant scores and a running per-row score as the camera moves.

Compared to the offline / batch pipeline (`VeXtronicsCode.ipynb`), this version differs in §7 onwards in two important ways:

1. **Row identity is declared by the operator.** Pressing `R` during the demo advances to the next row; pressing `0`–`9` jumps to that row directly. In a real tractor integration this would come from the tractor's GPS or row counter.
2. **The optimal line is built up monotonically and then locked.** This is the key correctness requirement.

## The line-building strategy

A single video can contain hundreds of frames of the same plant. We need a strategy that:

- Doesn't refit the line on every frame (jittery, expensive, and meaningless when the same plant is detected 30 times in a row)
- Doesn't let a plant influence the line that judges it (or the line could shift toward outliers, hiding their badness)
- Gives every plant in a row a consistent reference line

The approach:

- **First plant in a row:** the line is a vertical line through that plant's stem. (One point alone can't fix an angle.)
- **2nd, 3rd, 4th, 5th plants:** as each new plant is committed, refit the line with RANSAC over all committed stems so far. The line gradually stabilizes.
- **After the 5th plant:** the line is **locked**. New plants in the same row are scored against it but do *not* update it.
- **Row switch (R):** state resets for the new row.

This ensures every plant is scored against a reference that didn't depend on it (or, for plants 1-5, didn't depend on it much — a worth-flagging caveat is that the first few plants in a row influence their own reference line).

## Plant identity across frames

The detector reports the same physical plant on many consecutive frames. We must commit each *physical* plant to the row's stem list only once. Heuristic:

- A new detection is the "same plant" as the active candidate if its stem is within ~80 px and its bounding box overlaps significantly with the candidate's last bounding box.
- A candidate is committed to the row's stem list once it hasn't been seen for several frames (default: 6 frames, ≈0.2 s at 30 fps).

This is a lightweight tracker — no Kalman filter, no embedding similarity — but it's sufficient for walking-pace camera motion where plants don't move and the camera moves smoothly.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-row state and live optimal-line builder
# ─────────────────────────────────────────────────────────────────────────────

from dataclasses import dataclass, field
from typing import Optional

# How many committed plants are needed before the row's line is locked.
ROW_LOCK_AFTER_N_PLANTS = 5

# A new detection is the same physical plant as the active candidate if its
# stem is within this distance (pixels) AND its bbox IoU exceeds the threshold.
TRACK_SAME_PLANT_PX  = 80.0
TRACK_SAME_PLANT_IOU = 0.30

# A candidate is committed once it hasn't been seen for this many frames.
CANDIDATE_COMMIT_AFTER_GAP_FRAMES = 6


@dataclass
class CandidatePlant:
    """A plant being tracked across consecutive frames before being committed."""
    sum_xy: tuple = (0.0, 0.0)   # running sum of (x, y) over all observations
    n_obs: int = 0
    last_seen_frame: int = 0
    last_bbox: Optional[list] = None    # most recent bbox, for IoU matching

    def add_observation(self, stem_xy, bbox, frame_idx):
        self.sum_xy = (self.sum_xy[0] + stem_xy[0], self.sum_xy[1] + stem_xy[1])
        self.n_obs += 1
        self.last_seen_frame = frame_idx
        self.last_bbox = bbox

    @property
    def mean_xy(self):
        return (self.sum_xy[0] / self.n_obs, self.sum_xy[1] / self.n_obs)


@dataclass
class RowState:
    """All accumulated state for one row's traversal."""
    committed_stems: list = field(default_factory=list)   # list of (x, y)
    candidate: Optional[CandidatePlant] = None
    line: Optional['OptimalLine'] = None
    locked: bool = False
    status: str = "empty"


def _is_same_plant(detection, candidate):
    """
    Heuristic identity check. Same plant if stems are close AND bboxes overlap.
    Either alone is too loose; both together is robust for corn rows at
    walking-pace camera motion.
    """
    stem = detection.get('stem')
    if stem is None:
        return False

    # Stem proximity check
    dx = stem[0] - candidate.mean_xy[0]
    dy = stem[1] - candidate.mean_xy[1]
    if (dx * dx + dy * dy) ** 0.5 >= TRACK_SAME_PLANT_PX:
        return False

    # For the very first observation, candidate has no previous bbox — skip IoU
    if candidate.last_bbox is None:
        return True

    # Bbox overlap check (iou_xyxy was defined in §5.2)
    return iou_xyxy(detection['bbox'], candidate.last_bbox) >= TRACK_SAME_PLANT_IOU


def _refit_row_line(row_state, frame_shape, prior_angle_deg=90.0):
    """Recompute row_state.line from the committed stems."""
    H, W = frame_shape[:2]
    stems = row_state.committed_stems
    n = len(stems)

    if n == 0:
        row_state.line = OptimalLine.from_point_angle(W / 2, H / 2, prior_angle_deg)
        row_state.status = "prior (no plants yet)"
        return

    if n == 1:
        # One stem: vertical through it (1 point can't determine an angle).
        x, y = stems[0]
        row_state.line = OptimalLine.from_point_angle(x, y, prior_angle_deg)
        row_state.status = f"vertical through plant 1 (1/{ROW_LOCK_AFTER_N_PLANTS})"
        return

    # 2+ stems: RANSAC.
    try:
        row_state.line = OptimalLine.from_ransac(stems, residual_threshold=20.0)
        if n >= ROW_LOCK_AFTER_N_PLANTS:
            row_state.locked = True
            row_state.status = f"locked ({n} plants)"
        else:
            row_state.status = f"fitting ({n}/{ROW_LOCK_AFTER_N_PLANTS})"
    except Exception:
        row_state.status = f"fit failed (kept previous, {n} plants)"


def update_row_state_from_frame(row_state, plants, frame_idx, frame_shape):
    """
    Update one row's state given detections from the current frame.

    Tracks one plant at a time as a candidate. When the candidate hasn't been
    seen for several frames, commit it to the row's stem list (unless the row
    is locked) and refit the line.
    """
    # Try to match each visible plant to the active candidate
    matched_detection = None
    if row_state.candidate is not None:
        for p in plants:
            if _is_same_plant(p, row_state.candidate):
                matched_detection = p
                break

    if matched_detection is not None:
        # Candidate still in frame — update it
        row_state.candidate.add_observation(
            matched_detection['stem'], matched_detection['bbox'], frame_idx)
    elif row_state.candidate is not None:
        # Candidate not seen this frame — check whether to commit
        gap = frame_idx - row_state.candidate.last_seen_frame
        if gap >= CANDIDATE_COMMIT_AFTER_GAP_FRAMES:
            if not row_state.locked:
                row_state.committed_stems.append(row_state.candidate.mean_xy)
                _refit_row_line(row_state, frame_shape)
            row_state.candidate = None

    # If no active candidate, start one from any visible plant
    if row_state.candidate is None:
        for p in plants:
            if p.get('stem') is not None:
                new_cand = CandidatePlant()
                new_cand.add_observation(p['stem'], p['bbox'], frame_idx)
                row_state.candidate = new_cand
                break

    # Ensure the line exists for scoring (use prior on first call)
    if row_state.line is None:
        _refit_row_line(row_state, frame_shape)


def get_row_line(row_state):
    """Return the row's current optimal line and a status string for display."""
    return row_state.line, row_state.status

---
# 8. Optimal Seeding Line per Row Calculation

## The key insight

In our greenhouse trial, the seeds were deliberately hand-planted along straight perpendicular lines. So the **optimal line is the row they were supposed to land on**. We're not trying to discover it from the data; we're measuring how far the plants drifted from it.

## Two ways to supply the optimal line

1. **If you know exactly where the rows were planted** (e.g., from the physical layout or a marking line in the image): use `OptimalLine.from_point_angle()` and hard-code the values. This is the most accurate.

2. **If you don't**: fit a line through the detected stems in each row using **RANSAC**. RANSAC stands for Random Sample Consensus. Unlike ordinary least-squares fitting, it ignores outliers (the few plants that were planted badly off-line). Those outliers are exactly what we want to *score*, not fit our reference to.

In [ ]:
from dataclasses import dataclass
from sklearn.linear_model import RANSACRegressor

def angular_diff_deg(a, b):
    """
    Smallest unsigned difference between two axis angles, normalized to [0, 90].

    Because an axis has no direction (line, not arrow), 170 deg and -10 deg describe
    the same axis, and their difference should be 0.
    """
    if a is None or b is None:
        return None
    d = abs(a - b) % 180
    return min(d, 180 - d)

@dataclass
class OptimalLine:
    """
    Represents a line in image coordinates as a point + unit direction vector.

    Why store it this way instead of slope + intercept? Because slope-intercept
    breaks down for vertical lines (infinite slope). Point + direction works for
    any line orientation.

    Attributes
    ----------
    px, py : float
        A point on the line, in image pixels.
    dx, dy : float
        The direction vector of the line (unit length: dx*dx + dy*dy = 1).
    """
    px: float
    py: float
    dx: float
    dy: float

    @classmethod
    def from_point_angle(cls, px, py, angle_deg):
        """Construct a line from a point and an angle in degrees."""
        t = np.radians(angle_deg)
        return cls(px, py, np.cos(t), np.sin(t))

    @classmethod
    def from_ransac(cls, stems_xy, residual_threshold=15.0, random_state=SEED):
        """
        Fit a line to a cloud of stem (x, y) points using RANSAC.

        residual_threshold (in pixels) = how far from the line a point can be
        while still counting as an inlier. Set this to your planting tolerance.
        """
        stems = np.asarray(stems_xy)
        if len(stems) < 2:
            # Degenerate — only one point; return a horizontal line through it
            return cls(stems[0, 0], stems[0, 1], 1.0, 0.0)

        X, y = stems[:, 0:1], stems[:, 1]

        # Detect near-vertical rows and handle them separately.
        # A near-vertical row has very little x-spread — fitting y(x) is unstable.
        # We fit x(y) instead and then flip the result back.
        if X.std() < y.std() * 0.2:
            # Near-vertical case: regress x on y
            ry = RANSACRegressor(residual_threshold=residual_threshold,
                                 random_state=random_state).fit(stems[:, 1:2], stems[:, 0])
            m = ry.estimator_.coef_[0]     # slope of x = m*y + c
            c = ry.estimator_.intercept_   # intercept
            n = np.hypot(m, 1)              # magnitude for normalization
            # Direction vector (m, 1) normalized; passes through (c, 0)
            return cls(c, 0.0, m / n, 1.0 / n)
        else:
            # Normal case: regress y on x
            rx = RANSACRegressor(residual_threshold=residual_threshold,
                                 random_state=random_state).fit(X, y)
            m = rx.estimator_.coef_[0]      # slope of y = m*x + c
            c = rx.estimator_.intercept_
            n = np.hypot(1, m)
            # Direction vector (1, m) normalized; passes through (0, c)
            return cls(0.0, c, 1.0 / n, m / n)

    def perpendicular_distance(self, pt):
        """
        Signed perpendicular distance from a point to the line, in pixels.

        Uses the 2D cross-product formula:
          distance = (pt - P) x direction
        where x is the scalar cross product (positive = left of line, negative = right).

        We usually take the absolute value for scoring.
        """
        vx, vy = pt[0] - self.px, pt[1] - self.py
        return vx * self.dy - vy * self.dx

    def angle_deg(self):
        """Return the angle of the line in degrees, normalized to (-90, 90]."""
        a = np.degrees(np.arctan2(self.dy, self.dx))
        while a >   90: a -= 180
        while a <= -90: a += 180
        return a


def fit_optimal_lines(plants, known_lines=None):
    """
    Build a dict mapping row_id -> OptimalLine.

    Parameters
    ----------
    plants : list of plant dicts (with row_id set by assign_rows)
    known_lines : optional dict {row_id: (px, py, angle_deg)}
        If provided, hard-code the line for that row instead of fitting.
        Use this when you know the exact planted layout.
    """
    lines = {}

    # Group stems by row_id
    rows = defaultdict(list)
    for p in plants:
        if p['row_id'] >= 0 and p['stem'] is not None:
            rows[p['row_id']].append(p['stem'])

    for rid, stems in rows.items():
        if known_lines and rid in known_lines:
            # Hard-coded line from layout
            px, py, a = known_lines[rid]
            lines[rid] = OptimalLine.from_point_angle(px, py, a)
        else:
            # Fit with RANSAC from the detected stems
            lines[rid] = OptimalLine.from_ransac(stems)
    return lines


# --- Demo ---
optimal_lines = fit_optimal_lines(plants)
for rid, line in sorted(optimal_lines.items()):
    print(f"Row {rid}: angle={line.angle_deg():+.2f} deg  "
          f"passes through ({line.px:.0f}, {line.py:.0f})")

---
# 9. Alignment Scores and Deviations Computation

## Per-plant deviations

For every plant, we compute two numbers:

- **Positional deviation** — perpendicular pixel distance from the stem to its row's optimal line. Directly measures "how far off is this plant from where it was supposed to be?"
- **Angular deviation** — absolute angle between the plant's leaf axis and the expected orientation. For our perpendicular trial, leaves should grow **across** the row, so the expected leaf axis is 90° off from the row direction. `EXPECTED_LEAF_OFFSET_DEG = 90°`.

## Per-plant score (0 to 100, higher = better)

We turn each deviation into a score in [0, 100]:

```
pos_score(d) = 100 * exp(-d / POS_TOLERANCE_PX)         # exponential decay
ang_score(delta) = 100 * max(0, 1 - delta / ANG_TOLERANCE_DEG)  # linear decay, clamped at 0
plant_score = 0.5 * pos_score + 0.5 * ang_score         # equal weighting
```

Why exponential decay for position? A plant 30 pixels off is bad, but a plant 60 pixels off isn't "twice as bad" — it's "qualitatively wrong." Exponential decay matches that intuition.

Why linear for angle? Because angle has a natural maximum (90° = completely perpendicular = 0 score makes sense); pegging the score at 0 past the tolerance is the simplest defensible rule.

## Per-row score

Average the plant scores within each row. Also report the fraction of plants within tolerance — this is the number a farmer would actually care about: "85% of my plants in row 3 are within spec."

## Tuning knobs

The two constants below control how "strict" the scoring is. **Set these based on what a farmer considers acceptable planting error**, not what the model can achieve. The brief calls this out as an open question — we can only answer it once we see real planting data.

In [ ]:
# --- Scoring thresholds — tune these based on real-world planting tolerance ---
POS_TOLERANCE_PX  = 30.0      # pixel offset at which positional score drops to ~37/100 (1/e)
ANG_TOLERANCE_DEG = 20.0      # angle offset at which angular score hits 0/100
EXPECTED_LEAF_OFFSET_DEG = 90.0  # leaves grow across (perpendicular to) the row

def score_plants(plants, optimal_lines,
                 pos_tol=POS_TOLERANCE_PX,
                 ang_tol=ANG_TOLERANCE_DEG,
                 expected_offset=EXPECTED_LEAF_OFFSET_DEG):
    """
    Add scoring fields to each plant dict:
      pos_dev_px, ang_dev_deg, pos_score, ang_score, plant_score
    """
    for p in plants:
        rid = p['row_id']
        # Can't score plants with no row assignment or no stem
        if rid < 0 or rid not in optimal_lines or p['stem'] is None:
            p.update(dict(pos_dev_px=None, ang_dev_deg=None,
                          pos_score=None, ang_score=None, plant_score=None))
            continue

        line = optimal_lines[rid]

        # --- Positional deviation ---
        pos_dev = abs(line.perpendicular_distance(p['stem']))  # perpendicular distance, pixels
        p['pos_dev_px'] = pos_dev
        # Exponential decay: 100 at d=0, ~37 at d=pos_tol, ~14 at d=2*pos_tol
        p['pos_score']  = 100.0 * np.exp(-pos_dev / pos_tol)

        # --- Angular deviation ---
        if p['leaf_axis_deg'] is not None:
            # Expected leaf axis = row direction + 90 deg (perpendicular)
            expected_axis_deg = line.angle_deg() + expected_offset
            # Normalize expected axis to (-90, 90]
            while expected_axis_deg >   90: expected_axis_deg -= 180
            while expected_axis_deg <= -90: expected_axis_deg += 180

            ang_dev = angular_diff_deg(p['leaf_axis_deg'], expected_axis_deg)
            p['ang_dev_deg'] = ang_dev
            # Linear decay: 100 at dev=0, 0 at dev=ang_tol, still 0 beyond
            p['ang_score']   = 100.0 * max(0.0, 1.0 - ang_dev / ang_tol)
        else:
            p['ang_dev_deg'] = None
            p['ang_score']   = None

        # --- Combined score ---
        if p['ang_score'] is not None:
            p['plant_score'] = 0.5 * p['pos_score'] + 0.5 * p['ang_score']
        else:
            # If we couldn't compute angular (e.g., one leaf tip missing),
            # fall back to positional only
            p['plant_score'] = p['pos_score']
    return plants


def row_scores(plants):
    """
    Roll plant-level scores up to per-row summaries.

    For each row, returns:
      n              — number of plants
      mean_score     — average plant_score across the row
      mean_pos_px    — average positional deviation
      max_pos_px     — worst positional deviation (useful for spotting bad plants)
      mean_ang_deg   — average angular deviation
      within_tol_pct — fraction of plants within POS_TOLERANCE_PX
    """
    # Group plants by row
    rows = defaultdict(list)
    for p in plants:
        if p['row_id'] >= 0 and p['plant_score'] is not None:
            rows[p['row_id']].append(p)

    summary = {}
    for rid, plist in rows.items():
        pos_devs = [p['pos_dev_px']  for p in plist if p['pos_dev_px']  is not None]
        ang_devs = [p['ang_dev_deg'] for p in plist if p['ang_dev_deg'] is not None]
        scores   = [p['plant_score'] for p in plist]
        within   = sum(1 for d in pos_devs if d <= POS_TOLERANCE_PX) / max(1, len(pos_devs))

        summary[rid] = dict(
            n=len(plist),
            mean_score=float(np.mean(scores)),
            mean_pos_px=float(np.mean(pos_devs)) if pos_devs else None,
            max_pos_px=float(np.max(pos_devs)) if pos_devs else None,
            mean_ang_deg=float(np.mean(ang_devs)) if ang_devs else None,
            within_tol_pct=100.0 * within,
        )
    return summary


# --- Demo ---
plants = score_plants(plants, optimal_lines)
row_summary = row_scores(plants)

# Nice printout
print(f"{'Row':>4}  {'n':>3}  {'score':>6}  {'mean-pos':>9}  {'max-pos':>8}  "
      f"{'mean-ang':>9}  {'within-tol':>10}")
for rid, s in sorted(row_summary.items()):
    print(f"{rid:>4}  {s['n']:>3}  {s['mean_score']:>6.1f}  "
          f"{(s['mean_pos_px'] or 0):>9.1f}  {(s['max_pos_px'] or 0):>8.1f}  "
          f"{(s['mean_ang_deg'] or 0):>9.1f}  {s['within_tol_pct']:>9.1f}%")

---
# 10. Live demo — full alignment pipeline on a live camera feed

This is the deployment-style demo. The camera streams frames, the model runs inference per frame, the per-row state from §7 updates as plants are committed, and every detected plant gets scored against the current row's optimal line in real time. Per-row running scores accumulate across the session, and every plant observation is logged to a CSV for post-hoc analysis.

## Live controls

| Key | Action |
|-----|--------|
| Q / Esc | Quit |
| S | Save current annotated frame as a PNG |
| R | Switch to next row (increments row_id) |
| 0–9 | Set row_id directly to that digit |
| SPACE | Pause / resume |

## Camera setup

If your camera (built-in, USB, or Bluetooth-paired) is recognized by Windows as a webcam — visible in Settings → Bluetooth & devices → Cameras — it will be available to OpenCV via `cv2.VideoCapture(N)` where N is its index. To find your camera's index, run this snippet first in a separate cell:

```python
import cv2
for i in range(5):
    cap = cv2.VideoCapture(i)
    if cap.isOpened():
        ret, _ = cap.read()
        print(f"Index {i}: {'works' if ret else 'opens but no frames'}")
        cap.release()
    else:
        print(f"Index {i}: not available")
```

Set `CAMERA_INDEX` in the cell below to whichever index your camera shows up at.

## What you see on screen

- **White line** — the current row's optimal line (the seeding-line reference)
- **Color-coded bounding boxes** — green for high alignment score, yellow/red for low
- **Per-plant overlay** — score, positional deviation (px), angular deviation (°)
- **HUD top-left** — row id, plant count, FPS, current line status (`prior`, `vertical`, `fitting (N/5)`, `locked`)
- **Running row score** — mean score across all committed plants in this row, updated live

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Live alignment demo
# Prerequisites: §6 (detect_plants), §7 (RowState, update_row_state_from_frame,
#                get_row_line), §8 (OptimalLine), §9 (score_plants).
# ─────────────────────────────────────────────────────────────────────────────

import cv2
import time
import csv
from datetime import datetime
from collections import defaultdict

# ── Config ────────────────────────────────────────────────────────────────────
CAMERA_INDEX = 0                       # change to match your camera's index
CONF_THRESH  = 0.25
IOU_THRESH   = 0.45
WINDOW_NAME  = "Live Alignment Demo"

SAVE_DIR  = PROJECT_ROOT / "live_snapshots"
LIVE_CSV  = PROJECT_ROOT / "results" / "live_plants.csv"
SAVE_DIR.mkdir(exist_ok=True)
(PROJECT_ROOT / "results").mkdir(exist_ok=True)


# ── Drawing helpers ───────────────────────────────────────────────────────────
def color_by_score(score):
    """Map a score in [0, 100] to a BGR color (red → yellow → green)."""
    if score is None:
        return (128, 128, 128)
    g = int(np.clip(score * 2.55, 0, 255))
    r = int(np.clip((100 - score) * 2.55, 0, 255))
    return (0, g, r)


def draw_optimal_line(frame, line):
    H, W = frame.shape[:2]
    t = max(H, W) * 2
    p1 = (int(line.px - t * line.dx), int(line.py - t * line.dy))
    p2 = (int(line.px + t * line.dx), int(line.py + t * line.dy))
    cv2.line(frame, p1, p2, (255, 255, 255), 2, cv2.LINE_AA)


def annotate_frame(frame, plants, line, committed_stems):
    """Draw optimal line, committed stem markers, per-plant boxes and overlays."""
    draw_optimal_line(frame, line)

    # Mark committed stems with a small ring (so the operator can see what's
    # already "locked in" as part of the row's reference)
    for sx, sy in committed_stems:
        cv2.circle(frame, (int(sx), int(sy)), 8, (255, 255, 255), 2)

    for p in plants:
        score = p.get('plant_score')
        color = color_by_score(score)
        x1, y1, x2, y2 = map(int, p['bbox'])
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        if p['stem']:
            cv2.circle(frame, (int(p['stem'][0]), int(p['stem'][1])), 5, color, -1)
        if p['left_tip'] and p['right_tip']:
            lx, ly = map(int, p['left_tip']);  rx, ry = map(int, p['right_tip'])
            cv2.line(frame, (lx, ly), (rx, ry), color, 2, cv2.LINE_AA)
        lines = []
        if score is not None:                 lines.append(f"score: {score:.0f}")
        if p.get('pos_dev_px') is not None:   lines.append(f"pos: {p['pos_dev_px']:.0f}px")
        if p.get('ang_dev_deg') is not None:  lines.append(f"ang: {p['ang_dev_deg']:.0f}deg")
        for i, txt in enumerate(lines):
            cv2.putText(frame, txt, (x1, y1 - 8 - i * 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)


def draw_hud(frame, n_plants, fps, current_row, line_status,
             paused, row_running_score, n_committed):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (620, 130), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    state = "PAUSED" if paused else "LIVE"
    cv2.putText(frame, f"{state}  Row: {current_row}  Plants: {n_plants}  FPS: {fps:.1f}",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    cv2.putText(frame, f"Line: {line_status}  ({n_committed} committed)",
                (10, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 255), 1)
    if row_running_score is not None:
        cv2.putText(frame, f"Row {current_row} mean score: {row_running_score:.1f}",
                    (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    cv2.putText(frame, "Q/Esc=quit  S=save  R=next row  0-9=set row  SPACE=pause",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)


# ── Open camera ───────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(
        f"Cannot open camera index {CAMERA_INDEX}. "
        "Try other indices (0, 1, 2…) and confirm Windows recognises the camera."
    )
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print(f"Camera opened. Resolution: "
      f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x"
      f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print("Q/Esc=quit  S=save  R=next row  0-9=set row  SPACE=pause")

# ── State ─────────────────────────────────────────────────────────────────────
current_row   = 0
paused        = False
prev_time     = time.time()
frame_idx     = 0
display_frame = None
row_states    = defaultdict(RowState)                  # row_id → RowState
row_score_log = defaultdict(list)                      # row_id → list of plant scores

# Persistent CSV writer
csv_file   = open(LIVE_CSV, 'w', newline='')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['timestamp', 'frame_idx', 'row_id', 'plant_idx_in_frame',
                     'stem_x', 'stem_y', 'leaf_axis_deg',
                     'pos_dev_px', 'ang_dev_deg', 'plant_score',
                     'line_status', 'n_committed'])

try:
    while True:
        if not paused:
            ret, frame = cap.read()
            if not ret:
                print("Frame grab failed.")
                break
            frame_idx += 1

            # ── Inference ─────────────────────────────────────────────────
            plants = detect_plants(frame, conf=CONF_THRESH, iou=IOU_THRESH)

            # ── Update the active row's state (tracker + line refit) ──────
            row_state = row_states[current_row]
            update_row_state_from_frame(row_state, plants, frame_idx, frame.shape)
            line, status = get_row_line(row_state)

            # ── Score plants against the row's current line ───────────────
            for p in plants:
                p['row_id'] = current_row
            plants = score_plants(plants, {current_row: line})

            # ── CSV logging + running score accumulation ─────────────────
            ts = datetime.now().isoformat(timespec='seconds')
            n_committed = len(row_state.committed_stems)
            for idx, p in enumerate(plants):
                if p.get('plant_score') is not None:
                    row_score_log[current_row].append(p['plant_score'])
                csv_writer.writerow([
                    ts, frame_idx, current_row, idx,
                    f"{p['stem'][0]:.1f}" if p['stem'] else '',
                    f"{p['stem'][1]:.1f}" if p['stem'] else '',
                    f"{p['leaf_axis_deg']:.1f}" if p.get('leaf_axis_deg') is not None else '',
                    f"{p['pos_dev_px']:.1f}"   if p.get('pos_dev_px')   is not None else '',
                    f"{p['ang_dev_deg']:.1f}"  if p.get('ang_dev_deg')  is not None else '',
                    f"{p['plant_score']:.1f}"  if p.get('plant_score')  is not None else '',
                    status, n_committed,
                ])

            # ── Draw ──────────────────────────────────────────────────────
            annotate_frame(frame, plants, line, row_state.committed_stems)

            now = time.time()
            fps = 1.0 / max(now - prev_time, 1e-6)
            prev_time = now

            running = (np.mean(row_score_log[current_row])
                       if row_score_log[current_row] else None)
            draw_hud(frame, len(plants), fps, current_row, status,
                     paused, running, n_committed)

            display_frame = frame

        if display_frame is not None:
            cv2.imshow(WINDOW_NAME, display_frame)

        # ── Key handling ──────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key in (ord('q'), ord('Q'), 27):
            print("Quitting.")
            break
        elif key in (ord('s'), ord('S')) and display_frame is not None:
            ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
            path = SAVE_DIR / f"snapshot_row{current_row}_{ts_str}.png"
            cv2.imwrite(str(path), display_frame)
            print(f"Saved snapshot → {path}")
        elif key in (ord('r'), ord('R')):
            current_row += 1
            print(f"--- Switched to row {current_row} ---")
        elif key == ord(' '):
            paused = not paused
            print("Paused" if paused else "Resumed")
        elif ord('0') <= key <= ord('9'):
            current_row = key - ord('0')
            print(f"--- Row set to {current_row} ---")

finally:
    cap.release()
    cv2.destroyAllWindows()
    csv_file.close()

# ── Session summary ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Live demo session summary")
print("=" * 60)
print(f"{'Row':>4}  {'committed':>10}  {'observations':>13}  {'mean score':>11}  {'status':>20}")
for rid in sorted(row_states.keys() | row_score_log.keys()):
    rs = row_states.get(rid)
    scores = row_score_log.get(rid, [])
    n_committed = len(rs.committed_stems) if rs else 0
    status = rs.status if rs else ''
    mean = f"{np.mean(scores):.1f}" if scores else ''
    print(f"{rid:>4}  {n_committed:>10}  {len(scores):>13}  {mean:>11}  {status:>20}")
print(f"\nLive CSV: {LIVE_CSV}")
print(f"Snapshots: {SAVE_DIR}")

---
# 11. Closing notes — live demo specifics

## How each deliverable is addressed

| Deliverable | Where |
|-------------|-------|
| 1. Corn detection model | §4 trains `yolo11-pose`; §5 validates it |
| 2. Position + orientation per plant | §6 returns stem + leaf axis per plant |
| 3. Optimal growth line per row | §7 builds it per row, locked after 5 plants |
| 4. Deviation from optimal | §9 computes positional + angular deviation |
| 5. Plant and row alignment scores | §10 live, with CSV export |

## Live-specific tuning knobs

| Knob | Cell | What it controls |
|------|------|------------------|
| `ROW_LOCK_AFTER_N_PLANTS` | §7 | How many plants must be committed before the row's line is frozen. Lower = locks faster; higher = more averaging before lockdown. |
| `TRACK_SAME_PLANT_PX` | §7 | Stem distance threshold for "same plant" across frames. Raise if camera moves fast. |
| `TRACK_SAME_PLANT_IOU` | §7 | Bbox-overlap threshold for "same plant." Raise to be stricter. |
| `CANDIDATE_COMMIT_AFTER_GAP_FRAMES` | §7 | How long a plant must be unseen before it counts as having exited the frame. |

## Caveats to mention if asked

- **The first plant in a row gets a vertical-line reference**, which assumes the camera is aligned with the row direction. Reasonable for top-down deployment but a known limitation.
- **Plants 1–5 in a row partially influence their own reference line.** After plant 6, the line is locked and later plants are judged against a reference that didn't depend on them.
- **Pixel-to-cm conversion is not yet calibrated.** All deviations are in pixels. Add a single `PX_PER_CM` constant once camera calibration is done.
- **Row identity is manual** (R key). In tractor deployment this would come from GPS / row-counter telemetry.

## Companion notebook

For offline batch analysis of a folder of field photos (no live camera), see `VeXtronicsCode.ipynb`. It shares §1–§6 and §8–§9 with this notebook but replaces §7 and §10 with a folder-driven batch pipeline.